### I. Downloading the Dataset

In [ ]:
from torchvision.datasets import CIFAR10
from torchvision import transforms as T

transforms = T.Compose([
    T.ToTensor(),
])

dataset = CIFAR10(root='./data', train=True, download=True, transform=transforms)

In [ ]:
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, Dataset

def display_sample_image(image, label):
    plt.figure(figsize=(4, 4))
    plt.imshow(image.permute(1, 2, 0))
    plt.title(f'Label: {label}')
    plt.axis('off')
    plt.show()

dataloader = DataLoader(dataset, batch_size=64, shuffle=True)
for images, labels in dataloader:
    display_sample_image(images[0], dataset.classes[labels[0].item()])
    break

### II. Building the Model

In [ ]:
import torch.nn as nn
from torchvision.models import resnet18, ResNet18_Weights

class ResNet18CIFAR(nn.Module):
    def __init__(self, num_classes=100, pretrained=False):
        super().__init__()
        weights = ResNet18_Weights.IMAGENET1K_V1 if pretrained else None
        self.resnet = resnet18(weights=weights)

        # CIFAR tweaks
        self.resnet.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
        self.resnet.maxpool = nn.Identity()

        # classifier
        self.resnet.fc = nn.Linear(self.resnet.fc.in_features, num_classes)

    def forward(self, x):
        return self.resnet(x)


### III. Training Loop

In [ ]:
import torch.optim as optim
from torch.utils.data import random_split

epochs = 20
learning_rate = 0.001
model = ResNet18CIFAR(num_classes=10)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

train_size = int(len(dataset) * 0.7)
val_size = int(len(dataset) * 0.15)
test_size = len(dataset) - train_size - val_size

train_set, val_set, test_set = random_split(dataset, [train_size, val_size, test_size])
train_loader = DataLoader(train_set, batch_size=64, shuffle=True)
val_loader = DataLoader(val_set, batch_size=64, shuffle=False)
test_loader = DataLoader(test_set, batch_size=64, shuffle=False)

print("Train/Val/Test sizes:", len(train_set), len(val_set), len(test_set))

In [5]:
import torch
from tqdm import tqdm
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    optimizer.zero_grad()
    batch_idx = 0
    iterator = tqdm(train_loader, desc=f"Epoch [{epoch+1}/{epochs}]", unit="batch")
    for batch_idx, (images, labels) in enumerate(iterator):
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
        iterator.set_postfix(loss=running_loss / (batch_idx + 1))
    
    model.eval()
    val_loss = 0.0
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            val_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    val_loss /= len(val_loader)
    val_accuracy = 100 * correct / total
    print(f'Validation Loss: {val_loss:.4f}, Validation Accuracy: {val_accuracy:.2f}%')

Epoch [1/20]:   3%|▎         | 14/547 [00:17<11:11,  1.26s/batch, loss=2.25]


KeyboardInterrupt: 

### IV. Testing

In [ ]:
# Testing the model
model.eval()
test_loss = 0.0
correct = 0
total = 0
with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        loss = criterion(outputs, labels)
        test_loss += loss.item()
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
test_loss /= len(test_loader)
test_accuracy = 100 * correct / total
print(f'Test Loss: {test_loss:.4f}, Test Accuracy: {test_accuracy:.2f}%')

### V. Visualizing using t-sne

In [ ]:
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt

def plot_tsne(model, dataloader, num_samples=1000):
    model.eval()
    features = []
    labels = []
    with torch.no_grad():
        for images, label in dataloader:
            images = images.to(device)
            output = model.resnet(images)
            output = output.view(output.size(0), -1)
            features.append(output.cpu())
            labels.append(label)
            if len(features) * dataloader.batch_size >= num_samples:
                break
    features = torch.cat(features)[:num_samples]
    labels = torch.cat(labels)[:num_samples]
    
    tsne = TSNE(n_components=2, random_state=42)
    features_2d = tsne.fit_transform(features.numpy())
    
    plt.figure(figsize=(10, 10))
    scatter = plt.scatter(features_2d[:, 0], features_2d[:, 1], c=labels.numpy(), cmap='tab20', alpha=0.7)
    plt.legend(*scatter.legend_elements(), title="Classes")
    plt.title("t-SNE of ResNet18 Features")
    plt.show()

plot_tsne(model, test_loader, num_samples=1000)